<a href="https://colab.research.google.com/github/ha199/AI-Compliant-Routing-System-using-Machine-Learning-/blob/main/improved_version_Ai_complaint_routing_systemin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas scikit-learn sentence-transformers faiss-cpu openai-whisper opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.5 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=c89ae8ffae33a0356fdaa8f00731156f803a2fa6e9f89caecef5816fd642abde
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


Cell 2 — Import Libraries


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor

from sentence_transformers import SentenceTransformer
import faiss

import whisper
import cv2

In [4]:
from google.colab import files
uploaded = files.upload()

Saving complaints_dataset_500_rows.csv to complaints_dataset_500_rows.csv


Cell 3 — Load Dataset

In [5]:
df = pd.read_csv("complaints_dataset_500_rows.csv")

df = df.dropna()

print(df.shape)
df.head()

(500, 13)


,complaint_id,complaint_text,language,location_area,complaint_category,severity_score,citizen_history_count,department_label,officer_id,officer_specialization,priority_label,resolution_days,timestamp
0,C0001,Water has been leaking from an underground pip...,en,ZoneA,Water_Department,4,1,Water_Department,O118,Leak_Inspection,low,5,2024-04-04
1,C0002,Drainage is clogged and water is not flowing p...,en,ZoneB,Sewer_Department,3,0,Sewer_Department,O128,Sewage_Repair,low,5,2024-01-28
2,C0003,Sewer pipe appears blocked causing water backup,en,ZoneE,Sewer_Department,4,5,Sewer_Department,O136,Sewer_Cleaning,low,7,2024-03-30
3,C0004,Road surface is cracked and vehicles are slowi...,en,ZoneD,Road_Maintenance,5,0,Road_Maintenance,O138,Pothole_Fix,low,5,2024-03-30
4,C0005,Waste is scattered around the residential lane,en,ZoneD,Sanitation_Department,9,2,Sanitation_Department,O114,Waste_Collection,high,1,2024-01-12


Cell 4 — Encode Labels

In [6]:
dept_encoder = LabelEncoder()
priority_encoder = LabelEncoder()

df["dept_encoded"] = dept_encoder.fit_transform(df["department_label"])
df["priority_encoded"] = priority_encoder.fit_transform(df["priority_label"])

Cell 5 — Train/Test Split (Important to avoid leakage)

In [7]:
X_train, X_test, y_dept_train, y_dept_test = train_test_split(
    df["complaint_text"],
    df["dept_encoded"],
    test_size=0.2,
    random_state=42
)

_, _, y_priority_train, y_priority_test = train_test_split(
    df["complaint_text"],
    df["priority_encoded"],
    test_size=0.2,
    random_state=42
)

_, _, y_eta_train, y_eta_test = train_test_split(
    df["complaint_text"],
    df["resolution_days"],
    test_size=0.2,
    random_state=42
)

Cell 6 — Text Embeddings (Semantic Understanding)
Using Sentence Transformers so the model learns context instead of keywords.

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

X_train_emb = embedding_model.encode(X_train.tolist())
X_test_emb = embedding_model.encode(X_test.tolist())

Cell 7 — Department Prediction Model

In [9]:
dept_model = LogisticRegression(max_iter=2000)

dept_model.fit(X_train_emb, y_dept_train)

preds = dept_model.predict(X_test_emb)

print("Department Accuracy:", accuracy_score(y_dept_test, preds))
print(classification_report(y_dept_test, preds))

Department Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        24
           1       1.00      1.00      1.00        24
           2       1.00      1.00      1.00        17
           3       1.00      1.00      1.00        20
           4       1.00      1.00      1.00        15

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100



Cell 8 — Priority Prediction

In [10]:
priority_model = LogisticRegression(max_iter=2000)

priority_model.fit(X_train_emb, y_priority_train)

LogisticRegression(max_iter=2000)

Cell 9 — ETA Prediction

In [11]:
eta_model = RandomForestRegressor(n_estimators=200)

eta_model.fit(X_train_emb, y_eta_train)

RandomForestRegressor(n_estimators=200)

Cell 10 — Similar Complaint Search

In [12]:
all_embeddings = embedding_model.encode(df["complaint_text"].tolist())

dimension = all_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(all_embeddings))

Cell 11 — Similar Complaint Function

In [13]:
def find_similar(text, k=3):

    emb = embedding_model.encode([text])

    distances, idx = index.search(emb, k)

    return df.iloc[idx[0]][["complaint_text","department_label"]]

Cell 12 — Officer Routing
Important: officers restricted to predicted department.

In [14]:
def recommend_officers(department, k=3):

    officers = df[df["department_label"] == department]["officer_id"].unique()

    if len(officers) <= k:
        return list(officers)

    return list(np.random.choice(officers, k, replace=False))

Cell 13 — Whisper Audio Transcription

In [15]:
whisper_model = whisper.load_model("base")

def audio_to_text(audio_path):

    result = whisper_model.transcribe(audio_path)

    return result["text"]

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 82.4MiB/s]


Cell 14 — Video → Audio Extraction with OpenCV

In [16]:
def video_to_audio(video_path):

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)

    print("Video FPS:", fps)

    cap.release()

    return audio_to_text(video_path)

Cell 15 — Final Complaint Routing Pipeline

In [17]:
def route_complaint(text):

    emb = embedding_model.encode([text])

    dept_id = dept_model.predict(emb)[0]
    department = dept_encoder.inverse_transform([dept_id])[0]

    priority_id = priority_model.predict(emb)[0]
    priority = priority_encoder.inverse_transform([priority_id])[0]

    eta = int(eta_model.predict(emb)[0])

    officers = recommend_officers(department)

    similar = find_similar(text)

    return {
        "Complaint": text,
        "Department": department,
        "Recommended Officers": officers,
        "Priority": priority,
        "ETA Days": eta,
        "Similar Complaints": similar
    }

Cell 16 — Test Example


In [19]:
route_complaint("road damaged")

{'Complaint': 'road damaged',
 'Department': 'Road_Maintenance',
 'Recommended Officers': ['O132', 'O104', 'O120'],
 'Priority': 'low',
 'ETA Days': 4,
 'Similar Complaints':                                complaint_text  department_label
 13  Road damage is causing traffic congestion  Road_Maintenance
 30  Road damage is causing traffic congestion  Road_Maintenance
 78  Road damage is causing traffic congestion  Road_Maintenance}